# Chapter 8, Part 2: Agentic Recommender

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kimfalk/modern-recommender-systems/blob/main/notebooks/chapter-08/02_agentic_recommender.ipynb)

This notebook builds a ReAct agent that orchestrates the retrieval tools from Part 1,
then adds memory and conversational capabilities.

**What you'll build:**
- A ReAct agent with tool selection and multi-step reasoning
- Conversation memory (short-term and long-term)
- A multi-turn conversational recommender

**Prerequisites:** Run Part 1 first to have embeddings saved. 
**LLM requirement:** Agent reasoning requires a capable model. Use Gemini (free) or OpenAI, not a small local model.

In [ ]:
# Cell 1: Environment Setup
from recsys.utils.colab import setup_colab_environment, get_data_path, check_gpu

setup_colab_environment()
check_gpu()

In [ ]:
# Cell 2: Imports
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import json
import numpy as np
import pandas as pd
import faiss
from pathlib import Path

from recsys.data.loaders import load_movielens, load_movielens_descriptions
from recsys.fourstage_recsys.retrieval.itemknn_retrieval import ItemKNNRetrieval
from recsys.agentic.llm_client import LLMClient
from recsys.agentic.hybrid_retriever import HybridRetriever
from recsys.agentic.agent import MovieRecommenderAgent, Tool
from recsys.agentic.memory import (
  ConversationMemory, UserProfiler,
  ConversationalRecommender
)

DATA_PATH = get_data_path()

In [ ]:
# Cell 3: LLM Setup
# Agent reasoning requires a capable model.

# Option A (recommended, free): Google Gemini
# Get a free API key at https://aistudio.google.com/apikey
llm = LLMClient(
  backend="api",
  api_key=os.environ.get("GEMINI_API_KEY"),
  base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
  model="gemini-2.5-flash"
)

# Option B: OpenAI (paid, best quality)
# llm = LLMClient(
#   backend="api",
#   api_key=os.environ.get("OPENAI_API_KEY"),
#   model="gpt-4o-mini"
# )

# Quick test
print(llm.generate(
  system_prompt="Reply in one sentence.",
  user_message="What is a recommender system?"
))

In [ ]:
# Cell 4: Rebuild retriever from Part 1
ratings, movies = load_movielens(dataset='ml-25m', data_dir=DATA_PATH)
descriptions = load_movielens_descriptions(data_dir=DATA_PATH)
descriptions.columns = descriptions.columns.str.lower()

if 'overview' not in movies.columns and not descriptions.empty:
  movies = movies.merge(
    descriptions[['title', 'overview']].drop_duplicates('title'),
    on='title', how='left'
  )
  movies['overview'] = movies['overview'].fillna('')

embeddings = np.load(Path(DATA_PATH) / "movie_embeddings.npy")
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
emb_norm = embeddings.copy()
faiss.normalize_L2(emb_norm)
index.add(emb_norm.astype(np.float32))

ratings_sample = ratings.sample(n=min(500_000, len(ratings)), random_state=42)
item_knn = ItemKNNRetrieval(ratings_sample)

retriever = HybridRetriever(
  faiss_index=index, embeddings=emb_norm,
  movies_df=movies.reset_index(drop=True),
  item_knn=item_knn
)
print("Retriever ready.")

## 1. Building the ReAct Agent

We define tools that wrap the retriever, then let the LLM decide
which tool to call and when it has enough information to answer.

**Token budget matters:** Tools return compact results (title, genres, year)
to keep observations small. Detailed metadata can be fetched in a second
step if the LLM needs grounding context for explanations.

In [ ]:
# Cell 5: Define agent tools
# Return enough fields for the LLM to judge relevance
# (title + genres + year + a short overview snippet),
# but keep observations compact to stay within the
# context budget.

def search_movies(query, k=10):
  """Search for movies by natural language description."""
  results = retriever.search(query, k=int(k))
  out = []
  for r in results:
    overview = (r.get("overview") or "").strip()
    if len(overview) > 200:
      overview = overview[:200].rsplit(" ", 1)[0] + "..."
    out.append({
      "title": r["title"],
      "genres": r["genres"],
      "year": r.get("year"),
      "overview": overview,
    })
  return out

def filter_movies(movies, genre=None, 
                  year_min=None, year_max=None):
  """Filter a list of movies by metadata."""
  filtered = movies
  if genre:
    filtered = [
      m for m in filtered 
      if genre.lower() in m.get("genres", "").lower()
    ]
  if year_min:
    filtered = [
      m for m in filtered 
      if m.get("year") and m["year"] >= int(year_min)
    ]
  if year_max:
    filtered = [
      m for m in filtered 
      if m.get("year") and m["year"] <= int(year_max)
    ]
  return filtered

tools = [
  Tool(
    name="search_movies",
    description=(
      "Search for movies by description or similarity. "
      "Returns title, genres, year, and a short overview "
      "snippet you can use to judge relevance."
    ),
    function=search_movies,
    parameters={"query": "str", "k": "int (default 10)"}
  ),
  Tool(
    name="filter_movies",
    description=(
      "Filter a list of movies by genre, year_min, "
      "or year_max. Use after search to narrow results."
    ),
    function=filter_movies,
    parameters={
      "movies": "list of movie dicts",
      "genre": "str (optional)",
      "year_min": "int (optional)",
      "year_max": "int (optional)"
    }
  )
]

print(f"Defined {len(tools)} tools: {[t.name for t in tools]}")

In [ ]:
# Cell 6: Create and run the agent
agent = MovieRecommenderAgent(
  llm=llm, tools=tools, max_steps=5
)

response = agent.run(
  "Recommend movies like Beautiful Mind",
  debug=True
)
print("\n=== Final Response ===")
print(response)

In [ ]:
# Cell 7: Complex query requiring search + filter
response = agent.run(
  "Find me a sci-fi movie from the 1990s similar to Dune",
  debug=True
)
print("\n=== Final Response ===")
print(response)

In [ ]:
# Cell 8: Inspect the reasoning trace
trace = agent.get_trace()
print(f"Agent took {len(trace)} steps\n")
for step in trace:
  print(f"--- Step {step['step']} [{step['type']}] ---")
  if step["type"] == "action":
    print(f"  Thought: {step['thought'][:120]}")
    print(f"  Tool: {step['tool']}({step['args']})")
    obs = step['observation']
    if isinstance(obs, list):
      titles = [i['title'] for i in obs if isinstance(i, dict) and 'title' in i]
      print(f"  Returned: {titles[:5]}")
  elif step["type"] == "final_answer":
    print(f"  {step['content'][:200]}")
  elif step["type"] == "error":
    print(f"  {step.get('raw_response', '')[:150]}")

## 2. Adding Memory

### 2.1 Short-Term: Conversation Memory

The `ConversationMemory` keeps a sliding window of recent turns plus a
summary of older turns.

In [ ]:
# Cell 9: Conversation memory demo
memory = ConversationMemory(llm, max_recent_turns=4)

memory.add_turn("user", "I want a thriller")
memory.add_turn("assistant", "How about Se7en or Silence of the Lambs?")
memory.add_turn("user", "Too dark. Something more fun.")
memory.add_turn("assistant", "Try Knives Out or The Nice Guys.")

context = memory.get_context()
print(f"Context has {len(context)} messages:")
for msg in context:
  print(f"  [{msg['role']}] {msg['content'][:80]}")

### 2.2 Long-Term: User Taste Profiles

In [ ]:
# Cell 10: User profiler demo
profiler = UserProfiler(llm)

profiler.update_profile(
  user_id="user_42",
  conversation_summary=(
    "user: Recommend a thriller\n"
    "assistant: How about Se7en?\n"
    "user: Too dark. Something more fun.\n"
    "assistant: Try Knives Out.\n"
    "user: Perfect, I loved that!"
  )
)
print("Profile after session 1:")
print(profiler.get_profile("user_42"))

## 3. Conversational Recommender

The `ConversationalRecommender` combines retriever, LLM, memory,
and profiler into a multi-turn system.

In [ ]:
# Cell 11b: Reload recsys modules (so source edits take effect
# without restarting the kernel). Safe to re-run any time.
import sys, importlib
for _name in [m for m in list(sys.modules) if m.startswith("recsys.")]:
  del sys.modules[_name]
import recsys  # re-import top-level package
print("Reloaded recsys.* modules.")

In [ ]:
# Cell 11: Build the conversational recommender
conv_rec = ConversationalRecommender(
  llm=llm,
  retriever=retriever,
  profiler=profiler
)
print("Conversational recommender ready.")

In [ ]:
# Cell 12: Turn 1 - vague request
response = conv_rec.chat(
  user_id="user_42",
  user_message="I want to watch something tonight."
)
print(response)

In [ ]:
# Cell 13: Turn 2 - narrowing down
response = conv_rec.chat(
  user_id="user_42",
  user_message="Light and fun. But not dumb."
)
print(response)

In [ ]:
# Cell 14: Turn 3 - critiquing
response = conv_rec.chat(
  user_id="user_42",
  user_message="I've seen that one. Something light and fun?"
  
)
print(response)

In [ ]:
# Cell 14: Turn 3 - critiquing
response = conv_rec.chat(
  user_id="user_42",
  user_message="A random movie, I would like?"
)
print(response)

In [ ]:
# Cell 14: Turn 3 - critiquing
response = conv_rec.chat(
  user_id="user_42",
  user_message="why would I like that one?"
  
)
print(response)

In [ ]:
# Cell 15: End session - profile updates automatically
conv_rec.end_session("user_42")
print("Updated long-term profile:")
print(profiler.get_profile("user_42"))

## Summary

This notebook built:

1. **ReAct agent** with search and filter tools, multi-step reasoning, and inspectable traces
2. **Conversation memory** with sliding window and summarization
3. **User taste profiles** updated across sessions
4. **Conversational recommender** with elicitation, explanation, and critiquing

The next notebook (Part 3) covers evaluation: grounding validation, multi-turn tests, and LLM-as-judge.